# Epidemiological Module Redesign
## Towards Consistent and Reproducible Disease Modeling

### Objectives
- Standardize disease module architecture
- Improve code reuse and maintainability
- Enable better

# Current Challenges

## Inconsistent Implementation Patterns
- Each disease module (HIV, TB, Measles, etc.) has different structure
- Duplicated code across modules
- Inconsistent parameter handling
- Different approaches to risk modeling

## Limited Re

# Proposed Solution: Hierarchical Module Design

## Core Principles
1. **Declarative Disease Definition**: Configure diseases through structured definitions
2. **Standardized Components**: Common transmission, progression, and intervention patterns
3. **Type-based Architecture**: Distinguish communicable vs non-communicable diseases
4. **Modular Risk Models**: Build on existing linear model framework

In [ ]:
from abc import ABC, abstractmethod
from enum import Enum
from typing import List, Dict

class DiseaseType(Enum):
    COMMUNICABLE = "communicable"
    NON_COMMUNICABLE = "non_communicable"
    MIXED = "mixed"  # e.g., cancer with infectious causes

class EpidemiologicalModule(ABC):
    """Base class for all disease modules"""

    # Class attributes to be defined by subclasses
    DISEASE_TYPE: DiseaseType
    TRANSMISSION_MODEL: str = None  # Only for communicable diseases
    NATURAL_HISTORY_STAGES: List[str]
    RISK_FACTORS: List[str]

    def __init__(self, name=None):
        self.name = name
        self.risk_model = None
        self.progression_model = None
        self.transmission_model = None
        self.intervention_manager = None

    def pre_initialise_population(self, population):
        """Standardized setup for all diseases"""
        self._setup_risk_models()
        self._setup_progression_models()
        if self.DISEASE_TYPE in [DiseaseType.COMMUNICABLE, DiseaseType.MIXED]:
            self._setup_transmission_models()

    @abstractmethod
    def _setup_risk_models(self):
        """Setup linear models for risk factors"""
        pass

    @abstractmethod
    def _setup_progression_models(self):
        """Setup disease progression pathways"""
        pass

    def _setup_transmission_models(self):
        """Setup transmission (only for communicable diseases)"""
        if self.DISEASE

# Disease Type Specialization

## Communicable vs Non-Communicable Diseases

### Key Differences:
- **Communicable**: Require transmission models, mixing matrices, force of infection
- **Non-Communicable**: Focus on risk factors, progression, interventions
- **Mixed**: Elements of both (e.g., HPV → Cervical Cancer)

In [ ]:
class CommunicableModule(EpidemiologicalModule):
    """Base for infectious diseases (HIV, TB, Measles)"""
    DISEASE_TYPE = DiseaseType.COMMUNICABLE

    def __init__(self, name=None):
        super().__init__(name)
        self.mixing_matrix = None
        self.force_of_infection = None

    def _setup_transmission_models(self):
        self.mixing_matrix = self._build_mixing_matrix()
        self.force_of_infection = self._build_foi_model()

    @abstractmethod
    def _build_mixing_matrix(self):
        """Build age/risk-stratified mixing patterns"""
        pass

    @abstractmethod
    def _build_foi_model(self):
        """Build force of infection model"""
        pass

class NonCommunicableModule(EpidemiologicalModule):
    """Base for non-infectious diseases (Cervical Cancer, Epilepsy)"""
    DISEASE_TYPE

# Standardized Risk Model Framework

## Building

In [ ]:
class RiskModelManager:
    """Manages all risk factor models for a disease"""

    def __init__(self, disease_name: str):
        self.disease_name = disease_name
        self.models = {}
        self.risk_factors = []

    def add_linear_model(self, outcome: str, predictors: List[str],
                        interaction_terms: List[str] = None):
        """Add a linear model for a specific outcome"""
        self.models[outcome] = {
            'type': 'linear',
            'predictors': predictors,
            'interactions': interaction_terms or []
        }

    def add_survival_model(self, outcome: str, predictors: List[str]):
        """Add a survival model for time-to-event outcomes"""
        self.models[outcome] = {
            'type': 'survival',
            'predictors': predictors
        }

    def build_models(self, lm_manager):
        """Build all registered models using LinearModelManager"""
        for outcome, config in self.models.items():
            if config['type'] == 'linear':
                lm_manager.add_linear_model(
                    f"{self.disease_name}_{outcome}",
                    config['predictors'],
                    config.get('interactions', [])
                )

class StandardizedRiskFactors:
    """Common risk factors across diseases"""
    DEMOGRAPHIC = ['age_exact_years', 'sex', 'district']
    SOCIOECONOMIC = ['wealth_quintile', 'education_level']
    BEHAVIORAL = ['smoking', 'alcohol_use', 'physical

# Disease

In [ ]:
import numpy as np

class ProgressionModel:
    """Standardized disease progression modeling"""

    def __init__(self, states: List[str], transitions: Dict):
        self.states = states
        self.transitions = transitions
        self.rates = {}

    def add_transition_rate(self, from_state: str, to_state: str,
                          rate_function: callable):
        """Add transition rate between states"""
        self.rates[(from_state, to_state)] = rate_function

    def get_transition_probability(self, person_id: int, from_state: str,
                                 to_state: str, dt: float):
        """Calculate transition probability for a person"""
        if (from_state, to_state) not in self.rates:
            return 0.0

        rate = self.rates[(from_state, to_state)](person_id)
        return 1 - np.exp(-rate * dt)

class DiseaseStages:
    """Standard disease stage definitions"""

    # Communicable disease stages
    COMMUNICABLE_STAGES = [
        'susceptible', 'exposed', 'infectious',
        'recovered', 'deceased'
    ]

    # Non-communicable disease stages
    NCD_STAGES = [
        'healthy', 'at_risk', 'early_stage',
        'advanced_stage', 'terminal', 'deceased'
    ]

    # Chronic disease stages
    CHRONIC_STAGES = [
        'undiagnosed', 'diagnosed', 'treated',
        'controlled',

# Implementation Example: HIV Module

## Key Features:
- Communicable disease with sexual and vertical transmission
- Complex progression from acute to AIDS
- Multiple intervention points
- Co

In [ ]:
class AgeMixingMatrix:
    def __init__(self, age_groups, mixing_type):
        self.age_groups = age_groups
        self.mixing_type = mixing_type

class SexualTransmissionModel:
    def __init__(self, transmission_probability, partner_change_rate):
        self.transmission_probability = transmission_probability
        self.partner_change_rate = partner_change_rate

class Hiv(CommunicableModule):
    DISEASE_TYPE = DiseaseType.COMMUNICABLE
    TRANSMISSION_MODEL = "sexual_and_vertical"
    NATURAL_HISTORY_STAGES = ['susceptible', 'acute', 'chronic', 'aids', 'deceased']
    RISK_FACTORS = ['age_exact_years', 'sex', 'district', 'wealth_quintile']

    def __init__(self, name=None):
        super().__init__(name)
        self.parameters = {}
        self.lm = None

    def _setup_risk_models(self):
        self.risk_model = RiskModelManager('hiv')

        # Infection risk model
        self.risk_model.add_linear_model(
            'infection_risk',
            predictors=['age_exact_years', 'sex', 'district'],
            interaction_terms=['age_exact_years:sex']
        )

        # Progression models
        self.risk_model.add_survival_model(
            'time_to_aids',
            predictors=['age_at_infection', 'viral_load', 'cd4_count']
        )

        self.risk_model.build_models(self.lm)

    def _setup_progression_models(self):
        self.progression_model = ProgressionModel(
            states=self.NATURAL_HISTORY_STAGES,
            transitions={
                ('acute', 'chronic'): 'fixed_duration',
                ('chronic', 'aids'): 'survival_model',
                ('aids', 'deceased'): 'survival_model'
            }
        )

    def _build_mixing_matrix(self):
        # HIV-specific sexual mixing patterns
        return AgeMixingMatrix(
            age_groups=[(15,24), (25,34), (35,49), (50,100)],
            mixing_type='assortative'
        )

    def _build_foi_model(self):
        return SexualTransmissionModel(
            transmission_probability=self.parameters['transmission_prob'],
            partner_change_rate=self.parameters['partner_change_rate']
        )

In [ ]:
class CervicalCancer(NonCommunicableModule):
    DISEASE_TYPE = DiseaseType.MIXED  # Has infectious cause (HPV)
    NATURAL_HISTORY_STAGES = ['healthy', 'hpv_infection', 'precancer', 'cancer', 'deceased']
    RISK_FACTORS = ['age_exact_years', 'hiv_status', 'smoking', 'parity']

    def __init__(self, name=None):
        super().__init__(name)
        self.lm = None

    def _setup_risk_models(self):
        self.risk_model = RiskModelManager('cervical_cancer')

        self.risk_model.add_linear_model(
            'hpv_infection_risk',
            predictors=['age_exact_years', 'hiv_status', 'sexual_partners']
        )

        self.risk_model.add_linear_model(
            'progression_to_cancer',
            predictors=['hpv_type', 'hiv_status', 'smoking']
        )

        self.risk_model.build_models(self.lm)

    def _setup_progression_models(self):
        self.progression_model = ProgressionModel(
            states=self.NATURAL_HISTORY_STAGES,
            transitions={
                ('hpv_infection', 'precancer'): 'linear_model',
                ('precancer', 'cancer'): 'linear_model',
                ('cancer',

# Standardized Intervention Framework

## Common Intervention Categories:
- **Prevention**: Vaccines, behavioral interventions
- **Screening**: Testing and early detection

In [ ]:
class InterventionManager:
    """Manages all interventions for a disease"""

    def __init__(self):
        self.interventions = {
            'prevention': [],
            'screening': [],
            'treatment': [],
            'palliative': []
        }

    def register_intervention(self, intervention: 'StandardizedIntervention',
                            category: str):
        """Register an intervention in a category"""
        self.interventions[category].append(intervention)

    def get_eligible_interventions(self, person_id: int,
                                 disease_state: str):
        """Get interventions available for a person"""
        eligible = []
        for category, interventions in self.interventions.items():
            for intervention in interventions:
                if intervention.is_eligible(person_id, disease_state):
                    eligible.append(intervention)
        return eligible

class StandardizedIntervention(ABC):
    """Base class for all interventions"""

    def __init__(self, name: str, eligibility_criteria: Dict):
        self.name = name
        self.eligibility_criteria = eligibility_criteria
